# Transaction Foundation Model on Ray — Part 5: Batch embedding extraction

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

Part 4 trained the foundation model. The recurring production job is to run it over every card and store a fresh embedding for downstream models to consume. That's a heterogeneous job — CPU to read the tokens, GPU for the forward pass — and it streams through a single Ray Data pipeline.

In [ ]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts; mini = tiny, CPU-only
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

## Turn the trained model into one vector per window

The foundation model's value isn't its loss — it's the **embedding**: pool the encoder's hidden states into a single vector that summarizes a card's recent behavior, and hand that vector to whatever downstream model needs it (fraud in Part 6, but also churn, credit, personalization). Computing those vectors for every card on a schedule is the job that actually runs in production, over and over, as new transactions arrive.

We pool with `"last"` — the hidden state at the most recent transaction — because the eval windows are labeled per-transaction (is *this* transaction fraud?), and the last position is the readout aligned with that label. (For a whole-card summary you'd mean-pool instead.) Each embedding is written alongside its `label`, `split`, importance `weight`, and the target transaction's raw features, so Part 6 has everything in one place to compare raw-vs-FM-vs-fusion without re-reading anything.

## Embed every window with a pool of model replicas

This is the stage where Ray Data is genuinely differentiated, not just convenient. The forward pass wants a GPU, but the model is expensive to load — so loading it per batch would be hopeless. The fix is a **stateful** `map_batches`: `EmbeddingExtractor` is a callable class that loads the model **once per replica** in `__init__` and embeds each batch in `__call__`.

`ActorPoolStrategy(min_size=1, max_size=num_workers)` runs a pool of those replicas, growing from one up to `num_workers` as work demands; `num_gpus=1` places each on a GPU (and `num_cpus=1` keeps it on CPU at `mini`). The CPU Parquet read and the GPU forward pass run as one streamed pipeline — batches flow from the readers to the actors through the object store with no intermediate disk writes. Going from CPU here to a pool of GPU replicas at `full` is the `embed` config, not a code change; `scripts/04_extract_embeddings.py` runs this same `EmbeddingExtractor`.

In [ ]:
from src.embed import EmbeddingExtractor, embedding_health

eb = cfg["embed"]
print(f"embedding with up to {eb['num_workers']} replica(s), "
      f"{'GPU' if eb['use_gpu'] else 'CPU'}, batch_size={eb['batch_size']}")

# Re-runs reuse the cached embeddings.
if not os.path.exists(paths["embeddings"]):
    ds = ray.data.read_parquet(paths["tokenized_eval"])   # CPU read

    # Stateful map_batches: the model loads ONCE per replica (in __init__),
    # then embeds each batch in __call__ — not reloaded per batch.
    embedded = ds.map_batches(
        EmbeddingExtractor,
        fn_constructor_kwargs={"checkpoint_dir": paths["checkpoint"], "pooling": "last"},
        batch_size=eb["batch_size"],
        compute=ray.data.ActorPoolStrategy(min_size=1, max_size=eb["num_workers"]),  # 1..N replicas
        num_gpus=1 if eb["use_gpu"] else 0,                          # one GPU each when use_gpu
        num_cpus=1,                                                  # finite footprint (see Part 5 notes)
        batch_format="numpy",
    )
    embedded.write_parquet(paths["embeddings"])           # streamed write, no intermediate disk
    print(f"  wrote embeddings -> {paths['embeddings']}")
else:
    print(f"  reusing cached embeddings at {paths['embeddings']}")

## Did the embeddings actually learn anything?

The classic self-supervised failure is silent **representation collapse**: every customer maps to nearly the same vector while the loss looks fine. A cheap guard is to sample the embeddings and check two numbers — mean pairwise cosine similarity (→1.0 means collapse) and mean feature variance (→0 means collapse). `embedding_health` reports both.

In [ ]:
embedding_health(paths["embeddings"])

emb = pd.read_parquet(paths["embeddings"])
vec0 = np.asarray(emb["embedding"].iloc[0])       # Ray stores this as a tensor column
carried = [c for c in emb.columns if c != "embedding"]
print(f"\n{len(emb):,} window embeddings · dim {vec0.shape[-1]}")
print(f"carried alongside each vector (for Part 6): {carried}")

row = emb.iloc[0]
print(f"\nexample — card {int(row['card_id'])}, label {int(row['label'])}, split {row['split']}")
print(f"  embedding[:8] = {np.asarray(row['embedding'])[:8].round(3).tolist()}")

## Takeaways

**Ray Data**
- A stateful `map_batches` (a callable class + `ActorPoolStrategy`) loads the model **once per replica**, not once per batch — the difference between usable and hopeless for GPU inference.
- `num_gpus` on `map_batches` places each replica on a GPU; the CPU read and GPU forward pass run as one streamed pipeline with no intermediate disk. This heterogeneous CPU→GPU shape is where Ray Data genuinely earns its keep.
- The same code goes from one CPU replica at `mini` to a pool of GPU replicas at `full` by changing the `embed` config; the notebook runs the same `EmbeddingExtractor` as `scripts/04_extract_embeddings.py`.

**The embeddings**
- One pooled vector per window (`"last"` pooling = the most recent transaction's hidden state, aligned with the per-transaction fraud label).
- Each vector is written next to its `label`, `split`, `weight`, and raw target features, so the downstream stage needs no re-reads.
- A quick collapse check (mean pairwise cosine, feature variance) catches the silent self-supervised failure mode before it reaches the downstream model.

---

## Next

**Part 6 — Downstream fraud: raw vs FM vs fusion**: train the same XGBoost recipe on three feature sets — raw transaction fields, the FM embedding, and the two fused — and measure the lift at natural fraud prevalence with PR-AUC.